# MULTINOMIAL CLASSIFICATION 

#### Using the original (largely) barible for crime type, that contains __ categories (instead of 2)

In [1]:
# import libraries
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Train/test split
from sklearn.model_selection import train_test_split

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

# Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)

# Plotting (for ROC curves, feature importance, etc.)
import matplotlib.pyplot as plt


In [2]:
# read in the multiclass df
model_df_multiclass = pd.read_parquet("../data/model_df_multiclass.parquet")


In [7]:
model_df_multiclass.head()

,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,outcome_binary,employ_score_rate,income_score_rate,living_environment_score,barriers_score,outcome_multiclass_grouped
0,03b3bda6de7c51f6cd0820b7f998a1f878e7acbb6b18dd...,2025-01,West Yorkshire Police,-1.882755,53.924927,On or near Moorside Lane,E01010646,Bradford 001A,Burglary,Investigation complete; no suspect identified,0,0.047,0.056,28.436,24.793,Investigation complete; no suspect identified
1,6f9fe9a78c7ed106fadb658e4fd5313c1bb5bc5c8aa223...,2025-01,West Yorkshire Police,-1.878940,53.943744,On or near Cross End Fold,E01010646,Bradford 001A,Shoplifting,Investigation complete; no suspect identified,0,0.047,0.056,28.436,24.793,Investigation complete; no suspect identified
2,9cbc69a540485581b55d8d4723eb41b97d364bc98132d8...,2025-01,West Yorkshire Police,-1.872959,53.941733,On or near Cornerstones Close,E01010646,Bradford 001A,Vehicle crime,Investigation complete; no suspect identified,0,0.047,0.056,28.436,24.793,Investigation complete; no suspect identified
3,050a7cc33129f37f41cbbd38fe858e63e07dc239720be8...,2025-01,West Yorkshire Police,-1.883567,53.934131,On or near Cocking Lane,E01010646,Bradford 001A,Violence and sexual offences,Unable to prosecute suspect,0,0.047,0.056,28.436,24.793,Unable to prosecute suspect
4,b43bb9c614a189dd550a9e877b409cdb6449d6ec1c1073...,2025-01,West Yorkshire Police,-1.880138,53.945417,On or near Aynholme Drive,E01010646,Bradford 001A,Violence and sexual offences,Investigation complete; no suspect identified,0,0.047,0.056,28.436,24.793,Investigation complete; no suspect identified


In [8]:
model_df_multiclass.columns.tolist()

['crime_id',
 'month',
 'reported_by',
 'longitude',
 'latitude',
 'location',
 'lsoa_code',
 'lsoa_name',
 'crime_type',
 'last_outcome_category',
 'outcome_binary',
 'employ_score_rate',
 'income_score_rate',
 'living_environment_score',
 'barriers_score',
 'outcome_multiclass_grouped']

In [ ]:
# quick check to see if all the necessary variables are still there

print(X_train_multi.shape, X_test_multi.shape, y_train_multi.shape, y_test_multi.shape)
print(y_train_multi.value_counts(normalize=True))

In [3]:
# rebuild X_multi cleanly — only crime_type and income_score_rate should remain
X_multi = model_df_multiclass.drop(columns=[
    'crime_id',
    'month',
    'reported_by',
    'longitude',
    'latitude',
    'location',
    'lsoa_code',
    'lsoa_name',
    'last_outcome_category',
    'outcome_binary',
    'outcome_multiclass_grouped',
    'employ_score_rate',
    'living_environment_score',
    'barriers_score'
])

y_multi = model_df_multiclass['outcome_multiclass_grouped']

In [4]:
print(X_multi.columns.tolist())  # should show only ['crime_type', 'income_score_rate']

['crime_type', 'income_score_rate']


In [5]:
# Redo the test/train split

from sklearn.model_selection import train_test_split

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_multi
)



In [6]:
print(X_train_multi.shape, X_test_multi.shape)
print(X_test_multi.head())
print(y_train_multi.value_counts(normalize=True))

(182045, 2) (45512, 2)
                          crime_type  income_score_rate
11242      Criminal damage and arson              0.280
15726   Violence and sexual offences              0.087
205224  Violence and sexual offences              0.352
215672  Violence and sexual offences              0.437
46568                    Shoplifting              0.353
outcome_multiclass_grouped
Unable to prosecute suspect                            0.479497
Investigation complete; no suspect identified          0.443583
Local resolution                                       0.028411
Further action is not in the public interest           0.017562
Action to be taken by another organisation             0.016117
Offender given a caution                               0.008509
Further investigation is not in the public interest    0.004032
Formal action is not in the public interest            0.002192
Other                                                  0.000099
Name: proportion, dtype: float64


In [7]:
# check how many small category groups there are

print(y_test_multi.value_counts())

outcome_multiclass_grouped
Unable to prosecute suspect                            21823
Investigation complete; no suspect identified          20189
Local resolution                                        1293
Further action is not in the public interest             799
Action to be taken by another organisation               733
Offender given a caution                                 387
Further investigation is not in the public interest      183
Formal action is not in the public interest              100
Other                                                      5
Name: count, dtype: int64


In [8]:
# One-hot encode crime_type

# one-hot encode crime_type on training data
X_train_multi_encoded = pd.get_dummies(X_train_multi, columns=['crime_type'], drop_first=True)

# apply the same encoding to test data, aligning columns to match
X_test_multi_encoded = pd.get_dummies(X_test_multi, columns=['crime_type'], drop_first=True)
X_test_multi_encoded = X_test_multi_encoded.reindex(columns=X_train_multi_encoded.columns, fill_value=0)


drop_first=True avoids the dummy variable trap, 
 reindex  - so X_test_multi_encoded ends up with exactly the same columns, in the same order, as X_train_multi_encoded.

In [9]:
print(X_train_multi_encoded.shape[1] == X_test_multi_encoded.shape[1])
print(X_train_multi_encoded.columns.tolist())

True
['income_score_rate', 'crime_type_Burglary', 'crime_type_Criminal damage and arson', 'crime_type_Drugs', 'crime_type_Other crime', 'crime_type_Other theft', 'crime_type_Possession of weapons', 'crime_type_Public order', 'crime_type_Robbery', 'crime_type_Shoplifting', 'crime_type_Theft from the person', 'crime_type_Vehicle crime', 'crime_type_Violence and sexual offences']


In [10]:
# Scale income_score_rate

from sklearn.preprocessing import StandardScaler

scaler_multi = StandardScaler()

# fit on training data only, then transform it
X_train_multi_encoded['income_score_rate'] = scaler_multi.fit_transform(X_train_multi_encoded[['income_score_rate']])

# use the already-fitted scaler to transform test data (no fitting here)
X_test_multi_encoded['income_score_rate'] = scaler_multi.transform(X_test_multi_encoded[['income_score_rate']])

In [11]:
# Scale income_score_rate

from sklearn.preprocessing import StandardScaler

scaler_multi = StandardScaler()

# fit on training data only, then transform it
X_train_multi_encoded['income_score_rate'] = scaler_multi.fit_transform(X_train_multi_encoded[['income_score_rate']])

# use the already-fitted scaler to transform test data (no fitting here)
X_test_multi_encoded['income_score_rate'] = scaler_multi.transform(X_test_multi_encoded[['income_score_rate']])

used a separate scaler_multi object rather than reusing the binary pipeline's scaler — these are different datasets (different rows, since binary excluded the "excluded/censored" outcome rows that multiclass might handle differently), so they need their own independently-fitted scaler rather than sharing one fit on a different population.

In [12]:
print(X_train_multi_encoded['income_score_rate'].mean(), X_train_multi_encoded['income_score_rate'].std())
print(X_test_multi_encoded['income_score_rate'].mean(), X_test_multi_encoded['income_score_rate'].std())

1.2177721637900039e-17 1.000002746584965
-0.004535009410107379 1.0018674623097987


couldn't figure the above out so I looked it up - Train mean is effectively 0 (that tiny 1.3e-17 is just floating-point rounding noise, not a real deviation) and SD is 1.0, as expected since the scaler was fit directly on this data. Test mean (-0.0045) and SD (1.0019) are close to but not exactly 0/1 — which is the correct signature of no leakage, since the scaler's parameters came purely from train and were just applied, not refit, to test.

# LOGISTIC REGRESSION FOR MULTI CLASS

Notes: I didn't know that sklearn's LogisticRegression needs a multiclass strategy. The default in recent sklearn versions handles multiclass automatically via multinomial loss

also max_iter=1000 — multiclass logistic regression often needs more iterations to converge than binary, especially with 9 classes and a sizeable dataset. The sklearn default (max_iter=100) can throw a ConvergenceWarning otherwise. 

also I will try class_weight='balanced' to keep it consistent with binary 

In [13]:
from sklearn.linear_model import LogisticRegression

log_reg_multi = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
log_reg_multi.fit(X_train_multi_encoded, y_train_multi)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [14]:
# evaluate the multi class model 

# predicted class labels
y_pred_multi = log_reg_multi.predict(X_test_multi_encoded)

# predicted probabilities — now one column per class, not just a single column
y_pred_proba_multi = log_reg_multi.predict_proba(X_test_multi_encoded)

In [15]:
from sklearn.metrics import roc_auc_score, average_precision_score

roc_auc_multi = roc_auc_score(y_test_multi, y_pred_proba_multi, multi_class='ovr', average='macro')
print(f"ROC-AUC (macro, one-vs-rest): {roc_auc_multi:.4f}")

pr_auc_multi = average_precision_score(pd.get_dummies(y_test_multi), y_pred_proba_multi, average='macro')
print(f"PR-AUC (macro): {pr_auc_multi:.4f}")

ROC-AUC (macro, one-vs-rest): 0.7796
PR-AUC (macro): 0.2243


In [16]:
print(classification_report(y_test_multi, y_pred_multi))

/home/vscode/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vscode/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                                                     precision    recall  f1-score   support

         Action to be taken by another organisation       0.03      0.89      0.06       733
        Formal action is not in the public interest       0.01      0.39      0.01       100
       Further action is not in the public interest       0.00      0.00      0.00       799
Further investigation is not in the public interest       0.04      0.27      0.07       183
      Investigation complete; no suspect identified       0.75      0.42      0.54     20189
                                   Local resolution       0.69      0.63      0.66      1293
                           Offender given a caution       0.06      0.03      0.04       387
                                              Other       0.00      1.00      0.00         5
                        Unable to prosecute suspect       0.00      0.00      0.00     21823

                                           accuracy                 

/home/vscode/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [17]:
report_dict = classification_report(y_test_multi, y_pred_multi, output_dict=True)

small_classes = [
    'Formal action is not in the public interest',
    'Further action is not in the public interest',
    'Further investigation is not in the public interest',
    'Offender given a caution',
    'Other'
]

for cls in small_classes:
    print(cls, report_dict[cls])

/home/vscode/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vscode/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Formal action is not in the public interest {'precision': 0.005603448275862069, 'recall': 0.39, 'f1-score': 0.01104815864022663, 'support': 100.0}
Further action is not in the public interest {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 799.0}
Further investigation is not in the public interest {'precision': 0.04036243822075782, 'recall': 0.2677595628415301, 'f1-score': 0.07015032211882606, 'support': 183.0}
Offender given a caution {'precision': 0.06451612903225806, 'recall': 0.031007751937984496, 'f1-score': 0.041884816753926704, 'support': 387.0}
Other {'precision': 0.0019342359767891683, 'recall': 1.0, 'f1-score': 0.003861003861003861, 'support': 5.0}


/home/vscode/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [19]:
# repeat LR but remove the class=balanced to see if this was the issue

# Logistic Regression - multiclass, without class_weight='balanced'
lr_multi_unweighted = LogisticRegression(random_state=42)
lr_multi_unweighted.fit(X_train_multi_encoded, y_train_multi)

y_pred_multi_unweighted = lr_multi_unweighted.predict(X_test_multi_encoded)

print(classification_report(y_test_multi, y_pred_multi_unweighted))
print("Accuracy:", accuracy_score(y_test_multi, y_pred_multi_unweighted))

/home/vscode/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/vscode/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                                                     precision    recall  f1-score   support

         Action to be taken by another organisation       0.00      0.00      0.00       733
        Formal action is not in the public interest       0.00      0.00      0.00       100
       Further action is not in the public interest       0.00      0.00      0.00       799
Further investigation is not in the public interest       0.00      0.00      0.00       183
      Investigation complete; no suspect identified       0.75      0.67      0.71     20189
                                   Local resolution       0.69      0.63      0.66      1293
                           Offender given a caution       0.00      0.00      0.00       387
                                              Other       0.00      0.00      0.00         5
                        Unable to prosecute suspect       0.67      0.82      0.74     21823

                                           accuracy                 

/home/vscode/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vscode/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


NameError: name 'accuracy_score' is not defined